In [4]:
# read table data
from pygmid import Lookup as lk
import numpy as np
lv_nmos = lk('../lib/sg13_lv_nmos.mat')
lv_pmos = lk('../lib/sg13_lv_pmos.mat')
# list of parameters: VGS, VDS, VSB, L, W, NFING, ID, VT, GM, GMB, GDS, CGG, CGB, CGD, CGS, CDD, CSS, STH, SFL
# if not specified, minimum L, VDS=max(vgs)/2=0.9 and VSB=0 are used 

In [5]:
# 2. Design Specifications & Operating Points
VDD = 1.8
VCM = 1.2             # Output Common-Mode
V_OD = 0.35           # Differential Swing (350mV)
I_TAIL = 3.5e-3       # 3.5mA current requirement

# Node Voltage Assumptions for Headroom
V_TAIL_NMOS = 0.3     # Voltage at the source of the diff pair
V_TAIL_PMOS = 1.5     # Voltage at the source of the PMOS switches (VDD - 0.3)

# gm/ID Targets
GMID_SW = 7           # Low gm/ID for switching pair (M2, M3) to maximize speed
GMID_SRC = 12         # High gm/ID for current sources (M1, M4, M5) for high ro

# Length Selections
L_MIN = 0.18          # Min L for 1.8V HV devices in IHP130
L_SRC = 0.5           # Longer L for current sources to improve matching/stability

print(f"--- LVDS Transmitter Sizing Results (VDD={VDD}V, VCM={VCM}V) ---")

--- LVDS Transmitter Sizing Results (VDD=1.8V, VCM=1.2V) ---


In [3]:
# 3. Sizing NMOS Tail Current Source (M1)
# Use [0] to extract the value from the returned list/array
vgs_m1 = lv_nmos.lookup('VGS', GM_ID=GMID_SRC, L=L_SRC, VDS=0.3, VSB=0)[0]
id_w_m1 = lv_nmos.lookup('ID_W', GM_ID=GMID_SRC, L=L_SRC, VDS=0.3, VSB=0)[0]
w_m1 = I_TAIL / id_w_m1

#print(f"Tail NMOS (M1): W={round(float(w_m1), 2)}um, L={L_SRC}um, VGS={round(float(vgs_m1), 3)}V")

IndexError: list index out of range

In [ ]:
# 4. Sizing NMOS Differential Pair (M2, M3)
# Note: VSB=0.3 accounts for the source node voltage at 300mV
vgs_sw = lv_nmos.lookup('VGS', GM_ID=GMID_SW, L=L_MIN, VDS=0.9, VSB=0.3)[0]
id_w_sw = lv_nmos.lookup('ID_W', GM_ID=GMID_SW, L=L_MIN, VDS=0.9, VSB=0.3)[0]
w_sw = I_TAIL / id_w_sw

# Extracting FT and handling the list return
ft_sw_val = lv_nmos.lookup('FT', GM_ID=GMID_SW, L=L_MIN, VDS=0.9, VSB=0.3)[0]

print(f"Diff Pair (M2/M3): W={round(float(w_sw), 2)}um, L={L_MIN}um, VGS={round(float(vgs_sw), 3)}V")
print(f"Estimated Peak ft: {round(float(ft_sw_val)/1e9, 2)} GHz")

IndexError: list index out of range

In [ ]:
# 5. Sizing PMOS Top Current Sources (M4, M5)
# In DCS, top PMOS usually mirrors the total tail current
# VDS = VDD - V_TAIL_PMOS = 1.8 - 1.5 = 0.3V
vsg_m4 = lv_pmos.look_upVGS(GM_ID=GMID_SRC, L=L_SRC, VDS=0.3, VSB=0)
id_w_m4 = lv_pmos.lookup('ID_W', GM_ID=GMID_SRC, L=L_SRC, VDS=0.3, VSB=0)
w_m4 = I_TAIL / id_w_m4

print(f"Top PMOS (M4/M5): W={round(w_m4, 2)}um, L={L_SRC}um, VSG={round(float(vsg_m4), 3)}V")


--- NMOS Tail Current Source (M1) ---
VGS_bias = 0.422 V Vg = 1.378 V
Width = 485.66 um


In [ ]:
# 6. Check Bandwidth Constraint
# Assuming a load capacitance (C_load) of 1pF
f_3db = 1 / (2 * np.pi * 100 * 1e-12)
print(f"\nTarget System Bandwidth (with 1pF load): {round(f_3db/1e9, 2)} GHz")